# INSTANT-AI-TRADER — Solana low-cap alpha bot

Fill `.env` first. Set `DRY_RUN=1` until canary passes.

In [ ]:
import sys, os, asyncio, logging
import nest_asyncio
nest_asyncio.apply()

ROOT = os.path.abspath('.')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.config import load_config, load_keypair, load_smart_wallets
from src.bot import main, fetch_sol_price_usd, get_sol_balance
from src.logger import TradeLog

cfg = load_config()
print(f"dry_run={cfg.dry_run} capital_usd=${cfg.capital_usd} max_open={cfg.max_open_positions}")
print(f"helius_key_set={bool(cfg.helius_api_key)} keypair={cfg.keypair_path}")

## Sanity checks before launch

In [ ]:
import httpx
from solana.rpc.async_api import AsyncClient

async def _check():
    kp = load_keypair(cfg.keypair_path)
    client = AsyncClient(cfg.helius_http)
    http = httpx.AsyncClient()
    sol_price = await fetch_sol_price_usd(http)
    bal = await get_sol_balance(client, kp.pubkey())
    sw = load_smart_wallets()
    print(f"pubkey={kp.pubkey()}")
    print(f"sol_price_usd=${sol_price:.2f}  wallet_sol={bal:.5f}  capital_sol≈{cfg.capital_usd/sol_price:.5f}")
    print(f"smart_wallets_loaded={len(sw)}")
    if bal < cfg.sol_reserve + 0.01 and not cfg.dry_run:
        print('⚠️  Wallet below reserve. Fund before going live.')
    await http.aclose(); await client.close()

asyncio.get_event_loop().run_until_complete(_check())

## Launch the bot (background task)

In [ ]:
loop = asyncio.get_event_loop()
task = loop.create_task(main(cfg))
print('bot task started:', task)

## Live monitor — re-run this cell to refresh

In [ ]:
import pandas as pd
tl = TradeLog(cfg.db_path)
rows = asyncio.get_event_loop().run_until_complete(tl.tail(30))
df = pd.DataFrame(rows)
if not df.empty:
    df['ts'] = pd.to_datetime(df['ts'], unit='s')
df

## Realized PnL today

In [ ]:
pnl_sol = asyncio.get_event_loop().run_until_complete(tl.realized_pnl_today_sol())
print(f'realized_pnl_today: {pnl_sol:.5f} SOL')

## Stop the bot

In [ ]:
task.cancel()
try:
    asyncio.get_event_loop().run_until_complete(task)
except (asyncio.CancelledError, Exception) as e:
    print('stopped:', type(e).__name__)